# WavePID Analysis Example

This notebook demonstrates how to load and plot the ROOT output from the WavePID simulation.

## Prerequisites

Generate output first:
```bash
./OMSim_WavePID_study -n 10 --detector_type 3 --environment 2 -d 5 -e 30 -p mu- -o wavepid_example
```

Install Python dependencies:
```bash
pip install uproot awkward matplotlib numpy
```

In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import os

# === SET YOUR ROOT FILE PATH HERE ===
filename = "wavepid_example_hits.root"

if not os.path.exists(filename):
    raise FileNotFoundError(
        f"ROOT file not found: {os.path.abspath(filename)}\n\n"
        "Generate it first by running:\n"
        "  ./OMSim_WavePID_study -n 100 --detector_type 3 --environment 2 "
        "-r 5 -e 100 -p mu- -o wavepid_example\n\n"
        "Then either copy the output file here or update the 'filename' variable above."
    )

file = uproot.open(filename)
tree = file["PhotonHits"]
data = tree.arrays(library="np")

In [ ]:
print(f"Total photon hits: {len(data['hitTime'])}")
print(f"Available branches: {list(data.keys())}")
print(f"Number of events: {len(np.unique(data['eventID']))}")

## 1. Photon Origin Distribution

Bar chart showing the number of detected photons by origin category.

In [ ]:
origins = data["photonOrigin"]
unique_origins, counts = np.unique(origins, return_counts=True)

# Sort by count descending
sort_idx = np.argsort(-counts)
unique_origins = unique_origins[sort_idx]
counts = counts[sort_idx]

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(unique_origins)))
ax.bar(range(len(unique_origins)), counts, color=colors)
ax.set_xticks(range(len(unique_origins)))
ax.set_xticklabels(unique_origins, rotation=45, ha="right")
ax.set_ylabel("Number of photon hits")
ax.set_title("Photon Origin Distribution")
ax.set_yscale("log")

for i, (orig, cnt) in enumerate(zip(unique_origins, counts)):
    ax.text(i, cnt * 1.1, str(cnt), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## 2. Hit Time Distribution by Photon Origin

Stacked histogram of photon arrival times, colored by origin. This is the core analysis for WavePID.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

origin_colors = {
    "Cerenkov from Muon": "tab:blue",
    "Cerenkov from Electron": "tab:orange",
    "Cerenkov from Other": "tab:green",
    "Scintillation": "tab:red",
    "Bremsstrahlung": "tab:purple",
    "PrimaryOpticalPhoton": "tab:brown",
    "Other": "tab:gray",
}

hit_times = data["hitTime"]
time_bins = np.linspace(np.percentile(hit_times, 1), np.percentile(hit_times, 99), 100)

for origin_name in unique_origins:
    mask = origins == origin_name
    color = origin_colors.get(str(origin_name), "tab:gray")
    ax.hist(hit_times[mask], bins=time_bins, alpha=0.6, label=f"{origin_name} ({mask.sum()})",
            color=color, histtype="stepfilled")

ax.set_xlabel("Hit time (ns)")
ax.set_ylabel("Photon count")
ax.set_title("Photon Arrival Time Distribution by Origin")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 3. Wavelength Spectrum by Photon Origin

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

wavelengths = data["wavelength"]
wl_bins = np.linspace(300, 650, 80)

for origin_name in unique_origins:
    mask = origins == origin_name
    color = origin_colors.get(str(origin_name), "tab:gray")
    ax.hist(wavelengths[mask], bins=wl_bins, alpha=0.6, label=str(origin_name),
            color=color, histtype="stepfilled")

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Photon count")
ax.set_title("Wavelength Spectrum by Photon Origin")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 4. Per-Event Hit Count Distribution

In [ ]:
event_ids = data["eventID"]
unique_events, hits_per_event = np.unique(event_ids, return_counts=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(hits_per_event, bins=50, color="steelblue", edgecolor="black", alpha=0.7)
ax.set_xlabel("Photon hits per event")
ax.set_ylabel("Number of events")
ax.set_title(f"Hit Count Distribution ({len(unique_events)} events, mean={hits_per_event.mean():.1f} hits/event)")
plt.tight_layout()
plt.show()